# Group A — Structural Invariants

Verifies the three structural properties that must hold **at initialization and after every single demonstration**:

| Check | Condition | Meaning |
|---|---|---|
| **A1** | `∀ hU ∈ U_pre, ∀ hL ∈ L_pre : hU ⊆ hL` | U_pre is always more general than L_pre |
| **A2** | `∀ hL ∈ L_eff, ∀ hU ∈ U_eff : hL ⊆ hU` | L_eff is always contained in U_eff |
| **A3** | All four boundaries are non-empty | Version space has not collapsed |

Reference: `AlgorithmExplained.md §13 Group A`

In [1]:
import sys, os, contextlib, io, tempfile, shutil
# Resolve src/ from tests/
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

from unified_planning.shortcuts import *
from unified_planning.io import PDDLReader
import unified_planning
get_environment().credits_stream = None

from ascal.learner import Learner
from ascal.models import Literal, State, Action, Demonstration
from ascal.transitions import generate_lifted_demonstrations_from_problem

## Setup

In [2]:
MOCKUP_DIR   = os.path.join('..', 'benchmarks', 'mockup')
DOMAIN_FILE  = os.path.join(MOCKUP_DIR, 'domain.pddl')
PROBLEM_FILE = os.path.join(MOCKUP_DIR, 'problems', 'problem-00.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
assert os.path.exists(PROBLEM_FILE), f'Problem not found: {PROBLEM_FILE}'
print(f'Domain:  {DOMAIN_FILE}')
print(f'Problem: {PROBLEM_FILE}')

Domain:  ../benchmarks/mockup/domain.pddl
Problem: ../benchmarks/mockup/problems/problem-00.pddl


In [3]:
reader  = PDDLReader()
problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)

all_fluents    = problem.fluents
all_actions    = problem.actions
static_fluents = problem.get_static_fluents()
action_names   = [a.name for a in all_actions]

print('Actions :', action_names)
print('Fluents :', [f.name for f in all_fluents])
print('Static  :', [f.name for f in static_fluents])

Actions : ['pickup']
Fluents : ['on', 'on_table', 'clear', 'arm_empty', 'holding']
Static  : ['on']


## Helper — Invariant Checker

This function encodes all three Group A checks. It is called once at initialization
and again after every demonstration. Any failure raises an `AssertionError` immediately.

In [4]:
def check_group_a(learner, label=''):
    """Assert all three structural invariants for every action.

    A1: every U_pre hypothesis ⊆ every L_pre hypothesis
    A2: every L_eff hypothesis ⊆ every U_eff hypothesis
    A3: all four boundaries are non-empty (no collapse)

    Prints a one-line status per action.  Raises AssertionError on any violation.
    """
    tag = f'[{label}] ' if label else ''
    all_ok = True

    for a in learner.all_actions:
        n = a.name
        L_pre = learner.L_pre[n]
        U_pre = learner.U_pre[n]
        L_eff = learner.L_eff[n]
        U_eff = learner.U_eff[n]

        violations = []

        # A3 — non-emptiness (check first: A1/A2 need at least one element)
        if not L_pre: violations.append('A3: L_pre is EMPTY (collapsed)')
        if not U_pre: violations.append('A3: U_pre is EMPTY (collapsed)')
        if not L_eff: violations.append('A3: L_eff is EMPTY (collapsed)')
        if not U_eff: violations.append('A3: U_eff is EMPTY (collapsed)')

        if not violations:
            hL_pre = next(iter(L_pre))
            hL_eff = next(iter(L_eff))

            # A1 — every U_pre hypothesis ⊆ L_pre hypothesis
            for hU_pre in U_pre:
                if not hU_pre.issubset(hL_pre):
                    violations.append(
                        f'A1: U_pre hyp {set(hU_pre)} ⊄ L_pre hyp {set(hL_pre)}')

            # A2 — every U_eff hypothesis ⊇ L_eff hypothesis
            for hU_eff in U_eff:
                if not hL_eff.issubset(hU_eff):
                    violations.append(
                        f'A2: L_eff hyp {set(hL_eff)} ⊄ U_eff hyp {set(hU_eff)}')

        if violations:
            all_ok = False
            for v in violations:
                print(f'  {tag}✗ {n}: {v}')
        else:
            sizes = (f'L_pre={len(hL_pre)} U_pre_hyps={len(U_pre)} '
                     f'L_eff={len(hL_eff)} U_eff_hyps={len(U_eff)}')
            #print(f'  {tag}✓ {n}  [{sizes}]')

    if not all_ok:
        raise AssertionError(f'{tag}Group A invariants VIOLATED — see above')

print('check_group_a() defined.')

check_group_a() defined.


## Check A — At Initialization

Before any demonstration is processed, verify the initial boundary values are correct.

Expected state:
- `L_pre` = one hypothesis containing **all** lifted literals (most restrictive)
- `U_pre` = one hypothesis = **empty set** (most permissive — no preconditions)
- `L_eff` = one hypothesis = **empty set** (no effects known yet)
- `U_eff` = one hypothesis containing **all non-static** lifted literals

In [5]:
learner = Learner(all_fluents, all_actions, static_fluents)

print('=== Initialization state ===')
for a in all_actions:
    n = a.name
    hL_pre = next(iter(learner.L_pre[n]))
    hU_pre = next(iter(learner.U_pre[n]))
    hL_eff = next(iter(learner.L_eff[n]))
    hU_eff = next(iter(learner.U_eff[n]))
    print(f'\nAction: {n}')
    print(f'  L_pre ({len(hL_pre)} lits): {sorted(str(l) for l in hL_pre)}')
    print(f'  U_pre ({len(hU_pre)} lits): {sorted(str(l) for l in hU_pre)}')
    print(f'  L_eff ({len(hL_eff)} lits): {sorted(str(l) for l in hL_eff)}')
    print(f'  U_eff ({len(hU_eff)} lits): {sorted(str(l) for l in hU_eff)}')

print('\n=== Group A checks at initialization ===')
check_group_a(learner, label='init')

=== Initialization state ===

Action: pickup
  L_pre (8 lits): ['arm_empty()', 'clear(x)', 'holding(x)', 'on_table(x)', '¬arm_empty()', '¬clear(x)', '¬holding(x)', '¬on_table(x)']
  U_pre (0 lits): []
  L_eff (0 lits): []
  U_eff (8 lits): ['arm_empty()', 'clear(x)', 'holding(x)', 'on_table(x)', '¬arm_empty()', '¬clear(x)', '¬holding(x)', '¬on_table(x)']

=== Group A checks at initialization ===


## Generate Demonstrations

In [6]:
def generate_demos(problem):
    """Delegates to :func:`ascal.transitions.generate_lifted_demonstrations_from_problem`.

    ``max_neg_per_step=0`` means no per-step cap (all failing groundings kept).
    """
    return generate_lifted_demonstrations_from_problem(
        problem,
        max_neg_per_step=0,
        max_check_per_action=None,
        seed=0,
        verbose=True,
    )


tmpdir = tempfile.mkdtemp(prefix='ascal_test_')
orig_cwd = os.getcwd()
os.chdir(tmpdir)
try:
    pos_demos, neg_demos = generate_demos(problem)
finally:
    os.chdir(orig_cwd)
    shutil.rmtree(tmpdir, ignore_errors=True)

Plan: ['pickup(b2)']
Generated: 1 positive, 0 negative


## Check A — After Every Demonstration

Process every demo one at a time. After each one, run all three Group A checks.
Any violation raises immediately and stops the loop.

Demos are interleaved (proportional neg slice before each positive) as required by
the algorithm — see `AlgorithmExplained.md §13`.

In [7]:
# Re-initialize learner (clean slate)
learner = Learner(all_fluents, all_actions, static_fluents)

# Build interleaved sequence
if len(pos_demos) == 0:
    all_demos = list(neg_demos)
elif len(neg_demos) == 0:
    all_demos = list(pos_demos)
else:
    slice_size = len(neg_demos) / len(pos_demos)
    all_demos  = []
    for i, pos in enumerate(pos_demos):
        all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
        all_demos.append(pos)

print(f'Sequence: {len(all_demos)} demos '
      f'({sum(1 for d in all_demos if d.post_state) } pos, '
      f'{sum(1 for d in all_demos if not d.post_state)} neg)')
print(f'Order: [{" ".join("POS" if d.post_state else "NEG" for d in all_demos)}]\n')

passed = 0
failed = 0

for i, demo in enumerate(all_demos):
    kind = 'POS' if demo.post_state is not None else 'NEG'
    label = f'demo {i} [{kind}] {demo.action.name}'

    learner.update(demo)

    try:
        check_group_a(learner, label=label)
        passed += 1
    except AssertionError as e:
        print(f'\n❌ FAILED at {label}')
        failed += 1
        break   # stop on first violation

print(f'\n{"=" * 50}')
if failed == 0:
    print(f'✅ ALL {passed} checks PASSED — Group A invariants hold throughout learning.')
else:
    print(f'❌ FAILED after {passed} passing checks.')

Sequence: 1 demos (1 pos, 0 neg)
Order: [POS]


✅ ALL 1 checks PASSED — Group A invariants hold throughout learning.


## Final Boundary State

Print the boundaries after all demonstrations have been processed.

In [8]:
print('=== Final boundaries ===')
print(learner)

for a in all_actions:
    n = a.name
    hL_pre = next(iter(learner.L_pre[n]))
    hU_eff = next(iter(learner.U_eff[n]))
    hL_eff = next(iter(learner.L_eff[n]))

    print(f'\nAction: {n}')
    print(f'  L_pre ({len(hL_pre)} lits): {sorted(str(l) for l in hL_pre)}')
    print(f'  U_pre ({len(learner.U_pre[n])} hyps): '
          + str([sorted(str(l) for l in h) for h in learner.U_pre[n]]))
    print(f'  L_eff ({len(hL_eff)} lits): {sorted(str(l) for l in hL_eff)}')
    print(f'  U_eff ({len(hU_eff)} lits): {sorted(str(l) for l in hU_eff)}')
    print(f'  Converged: {learner.version_space_size[n]["converged"]}')

=== Final boundaries ===
Learner(actions=1, demos=1, collapsed=0, converged=False)

Action: pickup
  L_pre (4 lits): ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)']
  U_pre (1 hyps): [[]]
  L_eff (4 lits): ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬on_table(x)']
  U_eff (4 lits): ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬on_table(x)']
  Converged: False


## Check all problems of block domain (original)

In [9]:
# Setup
BENCH_DIR = os.path.join('..', 'benchmarks', 'blocks')
DOMAIN_FILE  = os.path.join(BENCH_DIR, 'domain_original.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
print(f'Domain:  {DOMAIN_FILE}')

Domain:  ../benchmarks/blocks/domain_original.pddl


In [10]:
for i in range(0, 20):
    PROBLEM_FILE = os.path.join(BENCH_DIR, 'problems', f"problem-{i:02d}.pddl")
    print(f"Checking problem {i}")
    
    problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)
    all_fluents = problem.fluents
    all_actions = problem.actions
    static_fluents = problem.get_static_fluents()
    learner = Learner(all_fluents, all_actions, static_fluents)
    
    print('\n=== Group A checks at initialization ===')
    check_group_a(learner, label='init')
    
    tmpdir = tempfile.mkdtemp(prefix='ascal_test_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        pos_demos, neg_demos = generate_demos(problem)
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)
    

    # Build interleaved sequence
    if len(pos_demos) == 0:
        all_demos = list(neg_demos)
    elif len(neg_demos) == 0:
        all_demos = list(pos_demos)
    else:
        slice_size = len(neg_demos) / len(pos_demos)
        all_demos  = []
        for i, pos in enumerate(pos_demos):
            all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
            all_demos.append(pos)

    #print(f'Sequence: {len(all_demos)} demos '
     #   f'({sum(1 for d in all_demos if d.post_state) } pos, '
     #  f'{sum(1 for d in all_demos if not d.post_state)} neg)')
    #rint(f'Order: [{" ".join("POS" if d.post_state else "NEG" for d in all_demos)}]\n')

    passed = 0
    failed = 0

    for i, demo in enumerate(all_demos):
        kind = 'POS' if demo.post_state is not None else 'NEG'
        label = f'demo {i} [{kind}] {demo.action.name}'

        learner.update(demo)

        try:
            check_group_a(learner, label=label)
            passed += 1
        except AssertionError as e:
            print(f'\n❌ FAILED at {label}')
            failed += 1
            break   # stop on first violation

    print(f'\n{"=" * 50}')
    if failed == 0:
        print(f'✅ ALL {passed} checks PASSED — Group A invariants hold throughout learning.')
    else:
        print(f'❌ FAILED after {passed} passing checks.')
        

Checking problem 0

=== Group A checks at initialization ===
Plan: ['unstack(b8, b3)', 'putdown(b8)', 'unstack(b3, b4)', 'putdown(b3)', 'unstack(b7, b2)', 'putdown(b7)', 'unstack(b2, b6)', 'putdown(b2)', 'unstack(b6, b5)', 'putdown(b6)', 'unstack(b5, b1)', 'stack(b5, b4)', 'pickup(b1)', 'stack(b1, b3)', 'pickup(b6)', 'stack(b6, b5)', 'pickup(b2)', 'stack(b2, b6)', 'pickup(b7)', 'stack(b7, b2)']
Generated: 20 positive, 2456 negative

✅ ALL 2476 checks PASSED — Group A invariants hold throughout learning.
Checking problem 1

=== Group A checks at initialization ===
Plan: ['unstack(b5, b3)', 'putdown(b5)', 'unstack(b7, b6)', 'stack(b7, b5)', 'unstack(b3, b2)', 'putdown(b3)', 'unstack(b2, b1)', 'putdown(b2)', 'pickup(b6)', 'stack(b6, b3)', 'unstack(b4, b8)', 'stack(b4, b1)', 'unstack(b7, b5)', 'putdown(b7)', 'pickup(b5)', 'stack(b5, b2)', 'pickup(b7)', 'stack(b7, b5)', 'unstack(b4, b1)', 'putdown(b4)', 'pickup(b1)', 'stack(b1, b7)', 'pickup(b4)', 'stack(b4, b1)']
Generated: 24 positive, 29

## Blocks - extended

In [11]:
# Setup
BENCH_DIR = os.path.join('..', 'benchmarks', 'blocks')
DOMAIN_FILE  = os.path.join(BENCH_DIR, 'domain_extended.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
print(f'Domain:  {DOMAIN_FILE}')

Domain:  ../benchmarks/blocks/domain_extended.pddl


In [12]:
for i in range(0, 20):
    PROBLEM_FILE = os.path.join(BENCH_DIR, 'problems', f"problem-{i:02d}.pddl")
    print(f"Checking problem {i}")
    
    problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)
    all_fluents = problem.fluents
    all_actions = problem.actions
    static_fluents = problem.get_static_fluents()
    learner = Learner(all_fluents, all_actions, static_fluents)
    
    print('\n=== Group A checks at initialization ===')
    check_group_a(learner, label='init')
    
    tmpdir = tempfile.mkdtemp(prefix='ascal_test_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        pos_demos, neg_demos = generate_demos(problem)
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)
    

    # Build interleaved sequence
    if len(pos_demos) == 0:
        all_demos = list(neg_demos)
    elif len(neg_demos) == 0:
        all_demos = list(pos_demos)
    else:
        slice_size = len(neg_demos) / len(pos_demos)
        all_demos  = []
        for i, pos in enumerate(pos_demos):
            all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
            all_demos.append(pos)

    #print(f'Sequence: {len(all_demos)} demos '
     #   f'({sum(1 for d in all_demos if d.post_state) } pos, '
     #  f'{sum(1 for d in all_demos if not d.post_state)} neg)')
    #rint(f'Order: [{" ".join("POS" if d.post_state else "NEG" for d in all_demos)}]\n')

    passed = 0
    failed = 0

    for i, demo in enumerate(all_demos):
        kind = 'POS' if demo.post_state is not None else 'NEG'
        label = f'demo {i} [{kind}] {demo.action.name}'

        learner.update(demo)

        try:
            check_group_a(learner, label=label)
            passed += 1
        except AssertionError as e:
            print(f'\n❌ FAILED at {label}')
            failed += 1
            break   # stop on first violation

    print(f'\n{"=" * 50}')
    if failed == 0:
        print(f'✅ ALL {passed} checks PASSED — Group A invariants hold throughout learning.')
    else:
        print(f'❌ FAILED after {passed} passing checks.')

Checking problem 0

=== Group A checks at initialization ===
Plan: ['unstack(b8, b3, b1)', 'putdown(b8, b1, b2)', 'unstack(b3, b4, b1)', 'putdown(b3, b1, b2)', 'unstack(b7, b2, b1)', 'putdown(b7, b1, b2)', 'unstack(b2, b6, b1)', 'putdown(b2, b1, b3)', 'unstack(b6, b5, b1)', 'putdown(b6, b1, b2)', 'unstack(b5, b1, b2)', 'stack(b5, b4, b1)', 'pickup(b1, b2, b3)', 'stack(b1, b3, b2)', 'pickup(b6, b1, b2)', 'stack(b6, b5, b1)', 'pickup(b2, b1, b3)', 'stack(b2, b6, b1)', 'pickup(b7, b1, b2)', 'stack(b7, b2, b1)']
Generated: 20 positive, 24672 negative

✅ ALL 24692 checks PASSED — Group A invariants hold throughout learning.
Checking problem 1

=== Group A checks at initialization ===
Plan: ['unstack(b5, b3, b1)', 'putdown(b5, b1, b2)', 'unstack(b7, b6, b1)', 'stack(b7, b5, b1)', 'unstack(b3, b2, b1)', 'putdown(b3, b1, b2)', 'unstack(b2, b1, b3)', 'putdown(b2, b1, b3)', 'pickup(b6, b1, b2)', 'stack(b6, b3, b1)', 'unstack(b4, b8, b1)', 'stack(b4, b1, b2)', 'unstack(b7, b5, b1)', 'putdown(b7, 

## Check all problems of Miconic domain (original)

In [13]:
# Setup
BENCH_DIR = os.path.join('..', 'benchmarks', 'blocks')
DOMAIN_FILE  = os.path.join(BENCH_DIR, 'domain_original.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
print(f'Domain:  {DOMAIN_FILE}')

Domain:  ../benchmarks/blocks/domain_original.pddl


In [14]:
for i in range(0, 20):
    PROBLEM_FILE = os.path.join(BENCH_DIR, 'problems', f"problem-{i:02d}.pddl")
    print(f"Checking problem {i}")
    
    problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)
    all_fluents = problem.fluents
    all_actions = problem.actions
    static_fluents = problem.get_static_fluents()
    learner = Learner(all_fluents, all_actions, static_fluents)
    
    print('\n=== Group A checks at initialization ===')
    check_group_a(learner, label='init')
    
    tmpdir = tempfile.mkdtemp(prefix='ascal_test_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        pos_demos, neg_demos = generate_demos(problem)
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)
    

    # Build interleaved sequence
    if len(pos_demos) == 0:
        all_demos = list(neg_demos)
    elif len(neg_demos) == 0:
        all_demos = list(pos_demos)
    else:
        slice_size = len(neg_demos) / len(pos_demos)
        all_demos  = []
        for i, pos in enumerate(pos_demos):
            all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
            all_demos.append(pos)

    #print(f'Sequence: {len(all_demos)} demos '
     #   f'({sum(1 for d in all_demos if d.post_state) } pos, '
     #  f'{sum(1 for d in all_demos if not d.post_state)} neg)')
    #rint(f'Order: [{" ".join("POS" if d.post_state else "NEG" for d in all_demos)}]\n')

    passed = 0
    failed = 0

    for i, demo in enumerate(all_demos):
        kind = 'POS' if demo.post_state is not None else 'NEG'
        label = f'demo {i} [{kind}] {demo.action.name}'

        learner.update(demo)

        try:
            check_group_a(learner, label=label)
            passed += 1
        except AssertionError as e:
            print(f'\n❌ FAILED at {label}')
            failed += 1
            break   # stop on first violation

    print(f'\n{"=" * 50}')
    if failed == 0:
        print(f'✅ ALL {passed} checks PASSED — Group A invariants hold throughout learning.')
    else:
        print(f'❌ FAILED after {passed} passing checks.')

Checking problem 0

=== Group A checks at initialization ===
Plan: ['unstack(b8, b3)', 'putdown(b8)', 'unstack(b3, b4)', 'putdown(b3)', 'unstack(b7, b2)', 'putdown(b7)', 'unstack(b2, b6)', 'putdown(b2)', 'unstack(b6, b5)', 'putdown(b6)', 'unstack(b5, b1)', 'stack(b5, b4)', 'pickup(b1)', 'stack(b1, b3)', 'pickup(b6)', 'stack(b6, b5)', 'pickup(b2)', 'stack(b2, b6)', 'pickup(b7)', 'stack(b7, b2)']
Generated: 20 positive, 2456 negative

✅ ALL 2476 checks PASSED — Group A invariants hold throughout learning.
Checking problem 1

=== Group A checks at initialization ===
Plan: ['unstack(b5, b3)', 'putdown(b5)', 'unstack(b7, b6)', 'stack(b7, b5)', 'unstack(b3, b2)', 'putdown(b3)', 'unstack(b2, b1)', 'putdown(b2)', 'pickup(b6)', 'stack(b6, b3)', 'unstack(b4, b8)', 'stack(b4, b1)', 'unstack(b7, b5)', 'putdown(b7)', 'pickup(b5)', 'stack(b5, b2)', 'pickup(b7)', 'stack(b7, b5)', 'unstack(b4, b1)', 'putdown(b4)', 'pickup(b1)', 'stack(b1, b7)', 'pickup(b4)', 'stack(b4, b1)']
Generated: 24 positive, 29

## Check drivelog - extended

In [15]:
# Setup
BENCH_DIR = os.path.join('..', 'benchmarks', 'blocks')
DOMAIN_FILE  = os.path.join(BENCH_DIR, 'domain_extended.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
print(f'Domain:  {DOMAIN_FILE}')

Domain:  ../benchmarks/blocks/domain_extended.pddl


In [16]:
for i in range(0, 20):
    PROBLEM_FILE = os.path.join(BENCH_DIR, 'problems', f"problem-{i:02d}.pddl")
    print(f"Checking problem {i}")
    
    problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)
    all_fluents = problem.fluents
    all_actions = problem.actions
    static_fluents = problem.get_static_fluents()
    learner = Learner(all_fluents, all_actions, static_fluents)
    
    print('\n=== Group A checks at initialization ===')
    check_group_a(learner, label='init')
    
    tmpdir = tempfile.mkdtemp(prefix='ascal_test_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        pos_demos, neg_demos = generate_demos(problem)
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)
    

    # Build interleaved sequence
    if len(pos_demos) == 0:
        all_demos = list(neg_demos)
    elif len(neg_demos) == 0:
        all_demos = list(pos_demos)
    else:
        slice_size = len(neg_demos) / len(pos_demos)
        all_demos  = []
        for i, pos in enumerate(pos_demos):
            all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
            all_demos.append(pos)

    #print(f'Sequence: {len(all_demos)} demos '
     #   f'({sum(1 for d in all_demos if d.post_state) } pos, '
     #  f'{sum(1 for d in all_demos if not d.post_state)} neg)')
    #rint(f'Order: [{" ".join("POS" if d.post_state else "NEG" for d in all_demos)}]\n')

    passed = 0
    failed = 0

    for i, demo in enumerate(all_demos):
        kind = 'POS' if demo.post_state is not None else 'NEG'
        label = f'demo {i} [{kind}] {demo.action.name}'

        learner.update(demo)

        try:
            check_group_a(learner, label=label)
            passed += 1
        except AssertionError as e:
            print(f'\n❌ FAILED at {label}')
            failed += 1
            break   # stop on first violation

    print(f'\n{"=" * 50}')
    if failed == 0:
        print(f'✅ ALL {passed} checks PASSED — Group A invariants hold throughout learning.')
    else:
        print(f'❌ FAILED after {passed} passing checks.')

Checking problem 0

=== Group A checks at initialization ===
Plan: ['unstack(b8, b3, b1)', 'putdown(b8, b1, b2)', 'unstack(b3, b4, b1)', 'putdown(b3, b1, b2)', 'unstack(b7, b2, b1)', 'putdown(b7, b1, b2)', 'unstack(b2, b6, b1)', 'putdown(b2, b1, b3)', 'unstack(b6, b5, b1)', 'putdown(b6, b1, b2)', 'unstack(b5, b1, b2)', 'stack(b5, b4, b1)', 'pickup(b1, b2, b3)', 'stack(b1, b3, b2)', 'pickup(b6, b1, b2)', 'stack(b6, b5, b1)', 'pickup(b2, b1, b3)', 'stack(b2, b6, b1)', 'pickup(b7, b1, b2)', 'stack(b7, b2, b1)']
Generated: 20 positive, 24672 negative

✅ ALL 24692 checks PASSED — Group A invariants hold throughout learning.
Checking problem 1

=== Group A checks at initialization ===
Plan: ['unstack(b5, b3, b1)', 'putdown(b5, b1, b2)', 'unstack(b7, b6, b1)', 'stack(b7, b5, b1)', 'unstack(b3, b2, b1)', 'putdown(b3, b1, b2)', 'unstack(b2, b1, b3)', 'putdown(b2, b1, b3)', 'pickup(b6, b1, b2)', 'stack(b6, b3, b1)', 'unstack(b4, b8, b1)', 'stack(b4, b1, b2)', 'unstack(b7, b5, b1)', 'putdown(b7, 